# 01 Driver and Executors

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand the roles of the <b>driver</b>, the <b>executors</b> and the <b>cluster manager</b>, which parts of your code run where, how cores, partitions and tasks relate, and how Spark Connect (serverless) changes the picture.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"Explain Spark's architecture" is one of the first questions in almost every Spark interview, and it explains most real problems: a <code>collect()</code> that crashes the driver, a job that uses only 4 of 64 cores, or a Python loop that runs on one machine while the cluster sits idle. This folder (<code>02_spark_internals</code>) moves you from "I can write PySpark" to "I understand how Spark executes my code".
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The three roles</b>
<ul><li><b>Driver</b>: the process that runs your <code>main</code> program (your notebook). It holds the SparkSession, turns your code into a plan, splits it into <b>jobs → stages → tasks</b>, schedules tasks, and collects results</li><li><b>Executors</b>: worker processes on the cluster nodes. Each executor has a number of <b>cores</b> (task slots) and memory. They run tasks and keep shuffle data and cached blocks</li><li><b>Cluster manager</b>: allocates resources for the executors (Standalone, YARN, Kubernetes, or Databricks' own manager)</li></ul>
<b>One task processes one partition on one core.</b> So the maximum parallelism is the total number of executor cores.
</div>

```
                  ┌──────────────────────────────┐
  your notebook ─▶│ DRIVER                       │
                  │  SparkSession, planner,      │
                  │  DAG scheduler, task scheduler│
                  └──────┬──────────────▲────────┘
            tasks        │              │ results / status
          ┌──────────────┼──────────────┼────────────┐
          ▼              ▼              ▼            ▼
   ┌────────────┐ ┌────────────┐ ┌────────────┐
   │ EXECUTOR 1 │ │ EXECUTOR 2 │ │ EXECUTOR 3 │      cluster manager
   │ core core  │ │ core core  │ │ core core  │      (allocates these)
   │ task task  │ │ task task  │ │ task task  │
   └────────────┘ └────────────┘ └────────────┘
     3 executors × 2 cores = 6 tasks run at the same time
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A team's nightly job reads 400 GB of logs, filters them, and then calls <code>.toPandas()</code> on the result to send a report. The filter runs in parallel on 20 executors in minutes, but <code>toPandas()</code> pulls 30 GB onto the <b>driver</b>, which runs out of memory and kills the job. Knowing what runs where, they aggregate on the executors first and only bring the 50-row summary to the driver.
</div>

## What runs where

| Code | Runs on | Why |
|---|---|---|
| Plain Python (`for` loops, `print`, `if`) | Driver | It's normal Python in your notebook process |
| Building DataFrames (`select`, `filter`, `join`) | Driver (planning only) | Only builds a plan, no data is touched |
| Built-in functions inside a plan (`F.upper`, `F.sum`) | Executors | Evaluated row by row inside tasks |
| Python UDFs and pandas UDFs | Executors (Python worker processes) | Shipped to the executors with the task |
| Results of `collect()`, `take()`, `toPandas()` | Driver | Data is sent back to the driver |
| `display()` / `show()` | Executors compute, driver receives a sample | Only the shown rows travel to the driver |

## On serverless: Spark Connect

```
 notebook (Python client)  ──plan over gRPC──▶  Spark server (driver)  ──tasks──▶  executors
            ▲                                         │
            └──────────────── Arrow result batches ───┘
```

On Free Edition your notebook is a **Spark Connect client**: it builds a plan and sends it to a Spark driver that Databricks runs and scales for you. You never see the executors, there is no `sparkContext`, and the Spark UI is replaced by the **query profile** (the "See performance" link under a cell's results). The concepts in this lesson are exactly the same, you just can't configure them.

## 1. The driver process

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints information about the Python process your notebook code runs in.<br><br>
<b>Why it matters</b><br>This is the driver side (on serverless, the Python client side). Everything that isn't a Spark expression runs here, one line after another.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A host name, a process ID and the CPU count of the driver-side Python process. Write them down.
</div>

In [0]:
import os, socket, platform

print("host       :", socket.gethostname())
print("process id :", os.getpid())
print("python     :", platform.python_version())
print("cpu count  :", os.cpu_count())

## 2. Code inside a UDF runs on the executors

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs a tiny Python UDF that reports <b>its own</b> host and process ID for each partition, and compares them with the driver's.<br><br>
<b>Why it matters</b><br>This makes "code is shipped to executors" visible. The UDF body is serialized, sent with each task, and executed in Python worker processes next to the executors, so it reports different process IDs (and usually different hosts) from the driver.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row per partition with the worker's host and process ID. The process IDs differ from the driver's. On a real cluster, the host names differ too.
</div>

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import udf

driver_pid = os.getpid()

@udf("string")
def where_am_i(_):
    import os, socket
    return f"{socket.gethostname()} / pid {os.getpid()}"

(
    spark.range(0, 8, 1, numPartitions=4)
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .agg(F.first(where_am_i("id")).alias("worker"), F.count("*").alias("rows"))
    .orderBy("partition")
    .show(truncate=False)
)
print("driver pid:", driver_pid)

## 3. Partitions are the unit of parallel work

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates DataFrames with different numbers of partitions and counts how many partitions (and therefore tasks) each one has, plus how many rows land in each.<br><br>
<b>Why it matters</b><br>Spark runs <b>one task per partition</b>. Two partitions on a 64-core cluster use only 2 cores. Thousands of tiny partitions waste time on scheduling. Partition counts are the first number to check when a job is slow (lesson 08 goes deeper).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The default <code>spark.range</code> partition count for your compute, then 2 and 12 partitions with roughly equal row counts per partition.
</div>

In [0]:
def partition_sizes(df):
    """Rows per partition, using only DataFrame functions (works on serverless)."""
    return df.groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition")

default_df = spark.range(1_000_000)
print("default partitions:", partition_sizes(default_df).count())

partition_sizes(spark.range(0, 1_000_000, 1, numPartitions=2)).show()
partition_sizes(spark.range(0, 1_000_000, 1, numPartitions=12)).show()

## 4. Python on the driver vs Spark on the executors

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sums the squares of 2 million numbers twice: once with a plain Python loop on the driver, and once as a Spark aggregation on the executors.<br><br>
<b>Why it matters</b><br>A Python loop only uses the driver's single process, no matter how big the cluster is. The same logic as a DataFrame expression is split into tasks that run in parallel. Writing the work as Spark operations is what makes it distributed.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Both results are identical. On a small number like this, the Python loop may even be faster, because Spark pays a fixed cost to plan and schedule a job. The difference is <b>scalability</b>: the loop can never use more than one core, while the Spark version spreads across every core in the cluster, so on real data volumes it wins by far.
</div>

In [0]:
import time

n = 2_000_000

start = time.time()
python_result = sum(i * i for i in range(n))          # one process, one core: the driver
python_time = time.time() - start

start = time.time()
spark_result = spark.range(n).select(F.sum(F.col("id") * F.col("id"))).first()[0]   # tasks on executors
spark_time = time.time() - start

print(f"python loop on driver : {python_result}  ({python_time:.2f}s)")
print(f"spark on executors    : {spark_result}  ({spark_time:.2f}s)")
print("same result:", python_result == spark_result)

## 5. Bringing data to the driver safely

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Compares returning a small aggregate, a limited sample, and the full data to the driver.<br><br>
<b>Why it matters</b><br><code>collect()</code> and <code>toPandas()</code> copy every row into the driver's memory. On big data that is the most common cause of driver out-of-memory errors. Aggregate or <code>limit</code> first, so only small results travel to the driver.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 1-row aggregate, then 5 rows from <code>take(5)</code>, then the size of a full collect of 100,000 rows. Never do the last one on large data.
</div>

In [0]:
big = spark.range(100_000).withColumn("value", F.rand(seed=7))

summary = big.agg(F.count("*").alias("rows"), F.avg("value").alias("avg_value")).collect()   # 1 row to the driver
print("aggregate first:", summary)

print("sample        :", big.take(5))                                                        # 5 rows to the driver

everything = big.collect()                                                                    # 100,000 rows to the driver
print("full collect  :", len(everything), "rows now held in driver memory")

## Under the hood

```
df.groupBy("k").count().show()

DRIVER                                          EXECUTORS
1. build logical plan (lazy)
2. action: optimize -> physical plan
3. DAGScheduler: split into stages at shuffles
4. TaskScheduler: one task per partition  ────▶  run tasks in parallel (1 task = 1 core)
                                                 write/read shuffle files between stages
5. receive results (here: 20 rows for show) ◀──  send results back
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> only actions make the driver schedule tasks on executors.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> transformations are recorded on the driver as a plan. No executor does anything yet.

**Failure handling:** if a task fails, the driver re-runs it (up to 4 attempts by default) on another executor. If an executor is lost, its tasks and any shuffle output it held are recomputed from the lineage. If the **driver** dies, the whole application fails, which is why driver memory matters so much.

The plan below shows what the driver will hand to the executors: two code-generated stages separated by an `Exchange` (the shuffle).

In [0]:
spark.range(1_000_000).groupBy((F.col("id") % 5).alias("k")).count().explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: driver out of memory / "Driver is temporarily unavailable"</b><br>
Too much data brought to the driver with <code>collect()</code>, <code>toPandas()</code> or a huge <code>display()</code>. Aggregate or <code>limit</code> first, or write results to a table.<br><br>
<b>⛔ Problem: a big cluster but the job uses only a few cores</b><br>Too few partitions. Check the partition count and repartition (lesson 08).<br><br>
<b>⛔ Problem: Python loop over rows is extremely slow</b><br>It runs on the driver only. Express the logic as DataFrame operations, or a pandas UDF if you need Python.<br><br>
**⛔ Problem: <code>sparkContext</code> / <code>defaultParallelism</code> not available on serverless**<br>Spark Connect has no driver-side context. Use DataFrame-based checks such as <code>spark_partition_id()</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Explain the Spark architecture.</b><br>
A Spark application has one driver and many executors. The driver runs the main program, builds the plan, splits it into jobs, stages and tasks, and schedules the tasks. Executors run on worker nodes, execute tasks in parallel on their cores, and hold shuffle data and cached blocks. A cluster manager such as YARN, Kubernetes or Databricks allocates the executor resources.<br><br>

<b>🎤 2. What does the driver do?</b><br>
It holds the SparkSession, analyzes and optimizes the query, creates the DAG of stages, schedules tasks on executors, tracks their progress, retries failures, and receives results of actions such as <code>collect</code>.<br><br>

<b>🎤 3. What is an executor?</b><br>
A JVM process on a worker node that runs tasks for one application. It has a fixed number of cores, which are task slots, and memory for execution, shuffle and caching.<br><br>

<b>🎤 4. How are cores, partitions and tasks related?</b><br>
Each task processes one partition on one core. The number of tasks in a stage equals the number of partitions, and the number of tasks running at once is limited by the total executor cores.<br><br>

<b>🎤 5. What happens if an executor fails?</b><br>
The driver reschedules its tasks on other executors. Lost shuffle or cached data is recomputed from the lineage. If the driver fails, the whole application fails.<br><br>

<b>🎤 6. Where does a Python UDF run?</b><br>
On the executors, in Python worker processes started next to each executor. The function is serialized on the driver and shipped with the tasks.<br><br>

<b>🎤 7. What is Spark Connect?</b><br>
A client-server architecture where the client sends logical plans over gRPC to a remote Spark driver and receives results as Arrow batches. Databricks serverless notebooks use it, so there is no local <code>sparkContext</code>.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A notebook calls <code>df.toPandas()</code> on a 200 GB DataFrame and the run fails with an out-of-memory error. Where did the memory run out?</b><br>
A. On every executor<br>
B. On the driver<br>
C. In the cluster manager<br>
D. In the Delta transaction log<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>toPandas()</code> and <code>collect()</code> move all rows to the driver.</details><br><br>

<b>Q2. A cluster has 4 workers with 8 cores each. A stage has 16 partitions. How many tasks of that stage can run at the same time?</b><br>
A. 4<br>
B. 8<br>
C. 16<br>
D. 32<br>
<details><summary><b>Show answer</b></summary><b>C.</b> There are 32 slots, but only 16 tasks exist (one per partition), so 16 run at once and half the cores sit idle.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a DataFrame of 3 million rows with exactly 6 partitions and show the row count per partition</li><li>Write a UDF that returns the worker process ID, and count how many distinct worker process IDs were used</li><li>Compute the average of <code>id * 2</code> without bringing more than one row to the driver</li><li>In two sentences, explain to an interviewer why <code>for row in df.collect(): ...</code> is a bad pattern on big data</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Six partitions and their sizes
six = spark.range(0, 3_000_000, 1, numPartitions=6)
partition_sizes(six).show()

# 2. Distinct worker processes
@udf("int")
def worker_pid(_):
    import os
    return os.getpid()

print("distinct worker pids:", six.select(worker_pid("id").alias("pid")).distinct().count())

# 3. Aggregate on the executors, one row back
print(six.agg(F.avg(F.col("id") * 2)).first()[0])

# 4. collect() copies every row into the driver's memory, which can crash it, and the loop then
#    runs on a single core, so the cluster's parallelism is wasted.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Driver: plans, schedules, collects results. Executors: run tasks on cores. Cluster manager: allocates resources</li><li>One task = one partition on one core. Parallelism is limited by partitions <b>and</b> cores</li><li>Plain Python runs on the driver. Spark expressions and UDFs run on executors</li><li><code>collect</code>, <code>toPandas</code>, <code>take</code> bring data to the driver. Aggregate or limit first</li><li>Failed tasks are retried, lost executors are recovered from lineage, a lost driver fails the app</li><li>Serverless = Spark Connect client. Same architecture, but no <code>sparkContext</code> and no executor tuning</li></ul>
</div>

| Task | Code |
|---|---|
| Rows per partition | `df.groupBy(F.spark_partition_id()).count()` |
| Set partitions at creation | `spark.range(0, n, 1, numPartitions=8)` |
| Small result to driver | `df.agg(...).first()`, `df.take(5)` |
| See the plan sent to executors | `df.explain()` |

## Git commit

```
git add 02_spark_internals/01_driver_executors.ipynb
git commit -m "add 02_01 driver executors notebook"
```